In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
#Get sklearn! https://scikit-learn.org/stable/
from sklearn import linear_model
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor # Make sure to import ensemble

In [ ]:
file_path = "/Users/danisanchez/Downloads/Waves_2023.txt"
df = pd.read_csv(file_path, sep=r'\s+')

In [ ]:
print(df.head())

In [ ]:
df = df.replace([99.0, 999.0, 99.00], np.nan)

# 2. Drop rows with missing values in your columns of interest
df = df.dropna(subset=['WVHT', 'DPD', 'APD', 'MWD', 'WTMP'])

X = df[['DPD', 'APD', 'MWD', 'WTMP']]  # Features
y = df['WVHT']                         # Target (Wave Height)

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Second split: Divide the remaining data into Train and Validation sets (75% / 25%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)

print(f"Train size: {X_train.shape}, Validation size: {X_val.shape}, Test size: {X_test.shape}")


In [ ]:
lr_model = LinearRegression()
dt_model = DecisionTreeRegressor(random_state=42)
rf_model = RandomForestRegressor(n_estimators=25, random_state=42, n_jobs=-1)

# 2. Fit models on training data
lr_model.fit(X_train, y_train)
dt_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

# 3. Helper function to calculate performance
def evaluate_model(model, name):
    train_mse = mean_squared_error(y_train, model.predict(X_train))
    val_mse = mean_squared_error(y_val, model.predict(X_val))
    test_mse = mean_squared_error(y_test, model.predict(X_test))
    print(f"{name} -> Train MSE: {train_mse:.4f} | Val MSE: {val_mse:.4f} | Test MSE: {test_mse:.4f}")

# 4. Print results
evaluate_model(lr_model, "Linear Regression (Baseline)")
evaluate_model(dt_model, "Decision Tree Regressor")
evaluate_model(rf_model, "Random Forest (25 Trees)")

In [ ]:
tree_counts = [2, 10, 25, 50, 100, 250, 500, 1000]
train_errors = []
val_errors = []

for n in tree_counts:
    rf = RandomForestRegressor(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    
    # Track errors
    train_errors.append(mean_squared_error(y_train, rf.predict(X_train)))
    val_errors.append(mean_squared_error(y_val, rf.predict(X_val)))

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(tree_counts, train_errors, marker='o', linestyle='-', label='Training Error', color='green')
plt.plot(tree_counts, val_errors, marker='s', linestyle='--', label='Validation Error', color='blue')
plt.xscale('log') # Makes visualization cleaner across large step scales
plt.xlabel('Number of Trees (Log Scale)')
plt.ylabel('Mean Squared Error (MSE)')
plt.title('Random Forest Performance vs. Number of Trees')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.show()